# BitTrend: Тестирование формул и валидация весов

Этап 6 plan.md — Jupyter для отладки:
- **6.1** Тестирование формул MVRV, NUPL, SOPR
- **6.2** Валидация весов и порогов

## 1. Импорты и настройка

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from bit_trend.scoring.calculator import (
    BitTrendScorer,
    WEIGHT_MVRV_Z, WEIGHT_NUPL, WEIGHT_SOPR,
    WEIGHT_MA200, WEIGHT_DERIVATIVES, WEIGHT_ETF,
    WEIGHT_MACRO, WEIGHT_FEAR_GREED,
)
from bit_trend.scoring.calculator import (
    _mvrv_z_score_to_component,
    _nupl_to_component,
    _sopr_to_component,
)
from bit_trend.portfolio.manager import PortfolioManager, SCORE_TO_BTC_PCT

## 2. Тестирование формул MVRV Z-Score

**Формула:** MVRV Z-Score = (Market Cap - Realized Cap) / StdDev(Market Cap)

**Интерпретация:**
- &lt; 0 — недооценка (хорошо для покупки)
- 0 … 3.5 — нейтральная зона
- &gt; 3.5 — переоценка (плохо для покупки)

In [2]:
mvrv_test_values = [-1.0, -0.5, 0.0, 1.0, 2.0, 3.5, 4.0, 5.0]
mvrv_results = [(v, _mvrv_z_score_to_component(v)) for v in mvrv_test_values]

df_mvrv = pd.DataFrame(mvrv_results, columns=['MVRV Z-Score', 'Component Score'])
print("MVRV Z-Score → Component Score (-100..+100):")
display(df_mvrv)

assert _mvrv_z_score_to_component(None) == 0.0, "None должен давать 0"
assert _mvrv_z_score_to_component(-1.0) > 0, "Недооценка должна давать положительный score"
assert _mvrv_z_score_to_component(6.0) < 0, "Сильная переоценка (MVRV>5.5) должна давать отрицательный score"
print("\n✓ MVRV тесты пройдены")

MVRV Z-Score → Component Score (-100..+100):


,MVRV Z-Score,Component Score
0,-1.0,80.000000
1,-0.5,90.000000
2,0.0,100.000000
3,1.0,42.857143
4,2.0,-14.285714
5,3.5,-100.000000
6,4.0,75.000000
7,5.0,25.000000



✓ MVRV тесты пройдены


## 3. Тестирование формулы NUPL

**NUPL** (Net Unrealized Profit/Loss) = (Market Cap - Realized Cap) / Market Cap

**Интерпретация:**
- &lt; 0 — капитуляция (хорошо для покупки)
- 0 … 0.75 — нейтральная зона
- &gt; 0.75 — эйфория (плохо для покупки)

In [3]:
nupl_test_values = [-0.2, -0.1, 0.0, 0.25, 0.5, 0.75, 0.9, 1.0]
nupl_results = [(v, _nupl_to_component(v)) for v in nupl_test_values]

df_nupl = pd.DataFrame(nupl_results, columns=['NUPL', 'Component Score'])
print("NUPL → Component Score (-100..+100):")
display(df_nupl)

assert _nupl_to_component(None) == 0.0, "None должен давать 0"
assert _nupl_to_component(-0.1) > 0, "Капитуляция должна давать положительный score"
assert -100 <= _nupl_to_component(0.9) <= 100, "NUPL component в диапазоне -100..+100"
print("\n✓ NUPL тесты пройдены")

NUPL → Component Score (-100..+100):


,NUPL,Component Score
0,-0.20,80.000000
1,-0.10,90.000000
2,0.00,100.000000
3,0.25,33.333333
4,0.50,-33.333333
5,0.75,-100.000000
6,0.90,40.000000
7,1.00,0.000000



✓ NUPL тесты пройдены


## 4. Тестирование формулы SOPR

**SOPR** (Spent Output Profit Ratio) — отношение цены продажи к цене покупки.

**Интерпретация:**
- &lt; 0.95 — продажа в убыток, часто дно (хорошо для покупки)
- 0.95 … 1.05 — нейтральная зона
- &gt; 1.05 — распределение, прибыльные продажи (плохо для покупки)

In [4]:
sopr_test_values = [0.8, 0.9, 0.95, 0.98, 1.0, 1.02, 1.05, 1.1]
sopr_results = [(v, _sopr_to_component(v)) for v in sopr_test_values]

df_sopr = pd.DataFrame(sopr_results, columns=['SOPR', 'Component Score'])
print("SOPR → Component Score (-100..+100):")
display(df_sopr)

assert _sopr_to_component(None) == 0.0, "None должен давать 0"
assert _sopr_to_component(0.9) > 0, "SOPR < 0.95 должен давать положительный score"
assert _sopr_to_component(1.1) < 0, "SOPR > 1.05 должен давать отрицательный score"
print("\n✓ SOPR тесты пройдены")

SOPR → Component Score (-100..+100):


,SOPR,Component Score
0,0.80,8.000000e+01
1,0.90,8.000000e+01
2,0.95,1.000000e+02
3,0.98,4.000000e+01
4,1.00,-8.526513e-14
5,1.02,-4.000000e+01
6,1.05,-1.000000e+02
7,1.10,-8.000000e+01



✓ SOPR тесты пройдены


## 5. Валидация весов метрик

Сумма весов должна быть 100%.

In [5]:
weights = {
    'MVRV Z-Score': WEIGHT_MVRV_Z,
    'NUPL': WEIGHT_NUPL,
    'SOPR': WEIGHT_SOPR,
    'MA200': WEIGHT_MA200,
    'Derivatives': WEIGHT_DERIVATIVES,
    'ETF': WEIGHT_ETF,
    'Macro': WEIGHT_MACRO,
    'Fear & Greed': WEIGHT_FEAR_GREED,
}

total = sum(weights.values())
df_weights = pd.DataFrame(list(weights.items()), columns=['Метрика', 'Вес'])
df_weights['Вес %'] = (df_weights['Вес'] * 100).round(1).astype(str) + '%'

print("Веса метрик:")
display(df_weights[['Метрика', 'Вес %']])
print(f"\nСумма весов: {total:.4f} ({'✓ OK' if abs(total - 1.0) < 0.001 else '⚠ Сумма ≠ 1.0 — итоговый score может выходить за [-100, +100]'})")

if abs(total - 1.0) >= 0.001:
    print(f"  Рекомендация: нормализовать веса (текущая сумма {total})")

Веса метрик:


,Метрика,Вес %
0,MVRV Z-Score,25.0%
1,NUPL,15.0%
2,SOPR,10.0%
3,MA200,15.0%
4,Derivatives,15.0%
5,ETF,15.0%
6,Macro,10.0%
7,Fear & Greed,5.0%



Сумма весов: 1.1000 (⚠ Сумма ≠ 1.0 — итоговый score может выходить за [-100, +100])
  Рекомендация: нормализовать веса (текущая сумма 1.1)


## 6. Валидация порогов score → сигнал

| Score | Сигнал |
|-------|--------|
| ≥ 50 | BUY |
| 10 … 49 | HOLD (накопление) |
| -10 … 9 | HOLD (осторожность) |
| -30 … -11 | REDUCE |
| < -30 | EXIT |

In [6]:
scorer = BitTrendScorer()

score_signal_tests = [
    (60, 'BUY'),
    (50, 'BUY'),
    (49, 'HOLD'),
    (10, 'HOLD'),
    (9, 'HOLD'),
    (-10, 'HOLD'),
    (-11, 'REDUCE'),
    (-30, 'REDUCE'),
    (-31, 'EXIT'),
    (-50, 'EXIT'),
]

results = []
for score, expected in score_signal_tests:
    got = scorer._score_to_signal(score)
    ok = '✓' if got == expected else '✗'
    results.append((score, expected, got, ok))

df_signal = pd.DataFrame(results, columns=['Score', 'Ожидается', 'Получено', 'OK'])
display(df_signal)

failed = [r for r in results if r[3] == '✗']
assert not failed, f"Ошибки в маппинге score→сигнал: {failed}"

,Score,Ожидается,Получено,OK
0,60,BUY,BUY,✓
1,50,BUY,BUY,✓
2,49,HOLD,HOLD,✓
3,10,HOLD,HOLD,✓
4,9,HOLD,HOLD,✓
5,-10,HOLD,HOLD,✓
6,-11,REDUCE,REDUCE,✓
7,-30,REDUCE,REDUCE,✓
8,-31,EXIT,EXIT,✓
9,-50,EXIT,EXIT,✓


## 7. Валидация порогов score → целевая аллокация BTC

| Score | BTC % в портфеле |
|-------|------------------|
| 70 … 100 | 95% |
| 50 … 69 | 80% |
| 30 … 49 | 65% |
| 10 … 29 | 50% |
| -10 … 9 | 40% |
| -29 … -11 | 25% |
| -49 … -30 | 15% |
| -100 … -50 | 5% |

In [7]:
pm = PortfolioManager()

score_btc_tests = [
    (100, 95), (70, 95),
    (69, 80), (50, 80),
    (49, 65), (30, 65),
    (29, 50), (10, 50),
    (9, 40), (-10, 40),
    (-11, 25), (-29, 25),
    (-30, 15), (-49, 15),
    (-50, 5), (-100, 5),
]

results = []
for score, expected_pct in score_btc_tests:
    got_pct = pm.get_target_btc_pct(score)
    ok = '✓' if got_pct == expected_pct else '✗'
    results.append((score, expected_pct, got_pct, ok))

df_btc = pd.DataFrame(results, columns=['Score', 'Ожидается BTC %', 'Получено BTC %', 'OK'])
display(df_btc)

failed = [r for r in results if r[3] == '✗']
assert not failed, f"Ошибки в маппинге score→BTC%: {failed}"

,Score,Ожидается BTC %,Получено BTC %,OK
0,100,95,95.0,✓
1,70,95,95.0,✓
2,69,80,80.0,✓
3,50,80,80.0,✓
4,49,65,65.0,✓
5,30,65,65.0,✓
6,29,50,50.0,✓
7,10,50,50.0,✓
8,9,40,40.0,✓
9,-10,40,40.0,✓


## 8. Интеграционный тест: полный расчёт score

Проверка на синтетических данных.

In [8]:
scorer = BitTrendScorer()

# Сценарий "бычий" — низкий MVRV, низкий NUPL, SOPR < 1
bullish_data = {
    'btc_price': 70000,
    'ma200': 65000,
    'mvrv_z_score': -0.5,
    'nupl': 0.2,
    'sopr': 0.92,
    'funding_rate': -0.0001,
    'open_interest_7d_change_pct': 5,
    'etf_flow_7d_usd': 300_000_000,
    'macro_signal': 1,
    'fear_greed_value': 30,
}

score_bull, signal_bull, comp_bull = scorer.compute(bullish_data)
print("Бычий сценарий:")
print(f"  Score: {score_bull}, Signal: {signal_bull}")
print("  Компоненты:", {k: round(v, 1) for k, v in comp_bull.items()})

# Сценарий "медвежий" — высокий MVRV, NUPL, SOPR
bearish_data = {
    'btc_price': 95000,
    'ma200': 70000,
    'mvrv_z_score': 4.0,
    'nupl': 0.85,
    'sopr': 1.08,
    'funding_rate': 0.0002,
    'open_interest_7d_change_pct': 15,
    'etf_flow_7d_usd': -400_000_000,
    'macro_signal': -1,
    'fear_greed_value': 80,
}

score_bear, signal_bear, comp_bear = scorer.compute(bearish_data)
print("\nМедвежий сценарий:")
print(f"  Score: {score_bear}, Signal: {signal_bear}")
print("  Компоненты:", {k: round(v, 1) for k, v in comp_bear.items()})

assert score_bull > score_bear, "Бычий score должен быть выше медвежьего"
assert signal_bull in ('BUY', 'HOLD'), "Бычий сценарий → BUY или HOLD"
assert signal_bear in ('REDUCE', 'EXIT'), "Медвежий сценарий → REDUCE или EXIT"
print("\n✓ Интеграционный тест пройден")

Бычий сценарий:
  Score: 65.8, Signal: BUY
  Компоненты: {'mvrv_z_score': 90.0, 'nupl': 46.7, 'sopr': 80.0, 'ma200': 30.8, 'derivatives': 50.0, 'etf': 48.0, 'macro': 50.0, 'fear_greed': 80.0}

Медвежий сценарий:
  Score: -18.4, Signal: REDUCE
  Компоненты: {'mvrv_z_score': 75.0, 'nupl': 60.0, 'sopr': -80.0, 'ma200': -50, 'derivatives': -80.0, 'etf': -64.0, 'macro': -50.0, 'fear_greed': -80.0}

✓ Интеграционный тест пройден


## 9. Опционально: загрузка реальных данных

Раскомментируйте для проверки на живых данных (требует .env с ключами).

In [9]:
# from dotenv import load_dotenv
# load_dotenv()
# from bit_trend.data.fetcher import DataFetcher
# fetcher = DataFetcher(ttl_seconds=0)
# data = fetcher.fetch_all(use_cache=False)
# score, signal, components = BitTrendScorer().compute(data)
# print(f"Score: {score}, Signal: {signal}")
# print("Components:", components)